In [ ]:
!pip install GEOparse

### Load NCBI GEO Data
`GEOparse` has download a Rheumatoid Arthritis dataset (e.g., GSE55235) and prepare it for the model.

In [ ]:
import GEOparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Downloading a common RA dataset (GSE55235 contains synovial tissue samples)
gse = GEOparse.get_GEO("GSE55235", destdir="./")

# Extract expression data
data = gse.pivot_samples('VALUE')

# For this demonstration, we transpose so rows are samples and columns are genes
data_t = data.T

# Mock labels: 0 for healthy/control, 1 for RA (In a real scenario, extract from gse.phenotype_data)
# Assuming the first half are controls and second half are RA for placeholder
labels = np.array([0] * (len(data_t)//2) + [1] * (len(data_t) - len(data_t)//2))

# Normalize
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_t)

# Reshape for RNN: [samples, time_steps, features]
# We'll treat gene sequences as temporal sequences for the RNN demo
X = data_scaled.reshape((data_scaled.shape[0], 1, data_scaled.shape[1]))
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

26-Apr-2026 11:10:08 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
26-Apr-2026 11:10:08 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE55nnn/GSE55235/soft/GSE55235_family.soft.gz to ./GSE55235_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE55nnn/GSE55235/soft/GSE55235_family.soft.gz to ./GSE55235_family.soft.gz
100%|██████████| 10.0M/10.0M [00:00<00:00, 20.3MB/s]
26-Apr-2026 11:10:09 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
26-Apr-2026 11:10:09 DEBUG downloader - Moving /tmp/tmpe4mh1ugg to /content/GSE55235_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpe4mh1ugg to /content/GSE55235_family.soft.gz
26-Apr-2026 11:10:09 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE55nnn/GSE55235/soft/GSE55235_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE55nnn/GSE55

### Define and Train RNN Model
Configure the model with specific units, learning rate, and epochs.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam

# CONFIGURATION
UNITS = 64
EPOCHS = 20
LEARNING_RATE = 0.001

model = Sequential([
    SimpleRNN(UNITS, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(1, activation='sigmoid')
])

optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_test, y_test))

# Display performance
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6667 - loss: 0.6989 - val_accuracy: 0.6667 - val_loss: 1.5274
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.6667 - val_loss: 2.0049
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 499ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 0.6667 - val_loss: 2.1460
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 579ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.6667 - val_loss: 2.2179
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step - accuracy: 1.0000 - loss: 8.9068e-04 - val_accuracy: 0.6667 - val_loss: 2.2660
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 1.0000 - loss: 8.0093e-04 - val_accuracy: 0.6667 - val_loss: 2.2994
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 678ms/step - accuracy: 1.0000 - loss: 7.4614e-04 - val_accuracy: 0.6667 - val_loss: 2.3235
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step - accuracy: 1.0000 - loss: 6.9779e-04 - val_accuracy: 0.6667 -